# PointPillars trên nuScenes — Box dự đoán vs Box nhãn gốc

Notebook chạy **PointPillars đã huấn luyện sẵn** (MMDetection3D) trên `v1.0-mini`, rồi hiển thị **hai hình 3D cạnh nhau**:

| Bên trái | Bên phải |
|---|---|
| **NHÃN GỐC** (ground truth) | **DỰ ĐOÁN** của PointPillars |
| Xanh lá = model tìm ra (TP) | Xanh lá = đúng (TP) |
| Cam = model bỏ sót (FN) | Đỏ = báo nhầm (FP) |

Kèm bảng so khớp theo **chuẩn đánh giá của nuScenes**: khớp khi cùng lớp và tâm cách nhau ≤ 2 m trên mặt phẳng ngang.

> ⚠️ **Bắt buộc bật GPU**: Runtime → Change runtime type → T4 GPU.

**Chuẩn bị:** upload `v1.0-mini.tgz` lên Google Drive.

## Cell 1 — Kiểm tra GPU

Nếu báo lỗi, vào Runtime → Change runtime type → chọn **T4 GPU** rồi chạy lại.

In [1]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

import sys
print('Python', sys.version.split()[0])
assert 'google.colab' in sys.modules or True, 'Notebook nay thiet ke cho Colab'

name, memory.total [MiB], driver_version
Tesla T4, 15360 MiB, 580.82.07
Python 3.13.15


## Cell 2 — Cài MMDetection3D

Đây là bước **lâu nhất (khoảng 10–15 phút)** và cũng dễ hỏng nhất, vì `mmcv` phải khớp chính xác phiên bản PyTorch.

Cách làm ở đây: ghim PyTorch **2.4.1 + CUDA 12.1**, rồi lấy `mmcv` bản dựng sẵn đúng cho tổ hợp đó. Bước cuối nới lỏng giới hạn phiên bản `mmcv` trong `mmdet3d` (mmdet3d 1.4.0 khai báo `mmcv < 2.2.0` nhưng thực tế chạy tốt với 2.2.0).

Sau khi chạy xong, Colab sẽ yêu cầu **RESTART SESSION** — hãy bấm nút đó rồi chạy tiếp từ Cell 3.

In [2]:
import os, subprocess, sys, time

ENV  = '/content/mmenv'           # moi truong Python 3.10 rieng cho MMDetection3D
PY   = f'{ENV}/bin/python'
LOG  = '/content/install.log'
DONE = f'{ENV}/.install_ok'


def run(desc, cmd):
    """Chay mot lenh cai dat; dung ngay va in log neu lenh that bai."""
    t0 = time.time()
    print(f'>> {desc} ...', end=' ', flush=True)
    with open(LOG, 'a') as f:
        f.write(f'\n\n$ {cmd}\n')
        f.flush()
        rc = subprocess.run(cmd, shell=True, stdout=f, stderr=subprocess.STDOUT).returncode
    if rc != 0:
        print('LOI\n')
        print(open(LOG).read()[-4000:])
        raise RuntimeError(f'Buoc "{desc}" that bai. Log day du: {LOG}')
    print(f'xong ({time.time() - t0:.0f}s)')


# 1) Kernel Colab: devkit de doc du lieu, plotly de ve, uv de tao moi truong rieng.
run('Cai nuscenes-devkit, plotly, uv vao kernel Colab',
    f'{sys.executable} -m pip install -q nuscenes-devkit plotly uv')

# 2) Moi truong Python 3.10 rieng cho MMDetection3D.
if os.path.exists(DONE):
    print('Moi truong MMDetection3D da cai truoc do, bo qua.')
else:
    UV = f'{sys.executable} -m uv pip install --python {PY}'
    MMCV_WHEELS = 'https://download.openmmlab.com/mmcv/dist/cu118/torch2.1.0/index.html'

    run('Tao moi truong Python 3.10',
        f'{sys.executable} -m uv venv {ENV} --python 3.10 --seed --clear')
    run('Cai PyTorch 2.1.2 + CUDA 11.8',
        f'{UV} torch==2.1.2 torchvision==0.16.2 --index-url https://download.pytorch.org/whl/cu118')
    # --no-build-package mmcv: bat buoc dung wheel dung san, khong bao gio build tu source.
    run('Cai mmengine + mmcv 2.1.0',
        f'{UV} "numpy<2" "mmengine>=0.10.3,<0.11" mmcv==2.1.0 '
        f'--find-links {MMCV_WHEELS} --no-build-package mmcv')
    run('Cai mmdet 3.3.0 + mmdet3d 1.4.0 + openmim',
        f'{UV} "numpy<2" mmdet==3.3.0 mmdet3d==1.4.0 openmim '
        f'--find-links {MMCV_WHEELS} --no-build-package mmcv')
    open(DONE, 'w').close()

print('\nCai dat xong. Chay tiep cell kiem tra ben duoi.')

>> Cai nuscenes-devkit, plotly, uv vao kernel Colab ... xong (12s)
>> Tao moi truong Python 3.10 ... xong (2s)
>> Cai PyTorch 2.1.2 + CUDA 11.8 ... xong (56s)
>> Cai mmengine + mmcv 2.1.0 ... LOI



$ /usr/bin/python3 -m pip install -q nuscenes-devkit plotly uv
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.5/20.5 MB 26.2 MB/s eta 0:00:00


$ /usr/bin/python3 -m uv venv /content/mmenv --python 3.10 --seed --clear
Using CPython 3.10.21
Creating virtual environment with seed packages at: mmenv
 + packaging==26.3
 + pip==26.2.1
 + setuptools==84.0.0
 + wheel==0.48.0
Activate with: source mmenv/bin/activate


$ /usr/bin/python3 -m uv pip install --python /content/mmenv/bin/python torch==2.1.2 torchvision==0.16.2 --index-url https://download.pytorch.org/whl/cu118
Using Python 3.10.21 environment at: mmenv
Resolved 18 packages in 1.05s
Prepared 18 packages in 53.95s
Installed 18 packages in 294ms
 + certifi==2022.12.7
 + charset-normalizer==2.1.1
 + filelock==3.32.3
 + fsspec==2026.7.0
 + id

RuntimeError: Buoc "Cai mmengine + mmcv 2.1.0" that bai. Log day du: /content/install.log

In [ ]:
import subprocess, sys

ENV = '/content/mmenv'
PY  = f'{ENV}/bin/python'

# Kiem tra moi truong MMDetection3D (Python 3.10).
CHECK = """
import sys, torch, numpy, mmengine, mmcv, mmdet, mmdet3d
from mmcv.ops import Voxelization          # op CUDA ma PointPillars can
print('python  ', sys.version.split()[0])
print('torch   ', torch.__version__, '| CUDA:', torch.cuda.is_available())
print('numpy   ', numpy.__version__)
print('mmengine', mmengine.__version__)
print('mmcv    ', mmcv.__version__)
print('mmdet   ', mmdet.__version__)
print('mmdet3d ', mmdet3d.__version__)
assert torch.cuda.is_available(), 'Chua bat GPU: Runtime -> Change runtime type -> T4 GPU'
"""
r = subprocess.run([PY, '-c', CHECK], capture_output=True, text=True)
print('--- Moi truong MMDetection3D ---')
print(r.stdout)
if r.returncode != 0:
    print(r.stderr[-4000:])
    raise RuntimeError('Moi truong MMDetection3D chua dung. Xem loi o tren, '
                       'hoac xoa /content/mmenv roi chay lai Cell 2.')

# Kiem tra kernel Colab.
import numpy, plotly, nuscenes
print('--- Kernel Colab ---')
print('python  ', sys.version.split()[0])
print('numpy   ', numpy.__version__)
print('plotly  ', plotly.__version__)
print('nuscenes-devkit OK')

CAI DAT THAT BAI: No module named 'mmengine'

--- Log cai dat (install_log) ---

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 811.7 kB/s eta 0:00:00
ERROR: Could not find a version that satisfies the requirement torchvision==0.20.1 (from versions: 0.1.6, 0.2.0)
ERROR: No matching distribution found for torchvision==0.20.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 818.2/818.2 kB 17.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
openxlab 0.1.3 requires setuptools~=60.2.0, but you have setuptools 84.0.0 which is incompatible.
pytensor 2.38.3 requires filelock>=3.15, but you have filelock 3.14.0 which is incompatible.
pytensor 2.38.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
pymc 5.28.5 requires rich>=13.7.1, but you have rich 13.4.2 which is incompatible.
torch 2.11.0+cu128 requires setuptools<82, but 

ModuleNotFoundError: No module named 'mmengine'

## Cell 3 — Kiểm tra cài đặt

Chạy sau khi đã restart. Nếu cell này lỗi, xem bảng xử lý sự cố ở cuối notebook.

In [ ]:
# import torch, mmcv, mmdet, mmdet3d, numpy as np

# print('torch   ', torch.__version__, '| CUDA:', torch.cuda.is_available())
# print('mmcv    ', mmcv.__version__)
# print('mmdet   ', mmdet.__version__)
# print('mmdet3d ', mmdet3d.__version__)
# print('numpy   ', np.__version__)
# assert torch.cuda.is_available(), 'Chua bat GPU: Runtime -> Change runtime type -> T4 GPU'

## Cell 4 — Dữ liệu nuScenes từ Google Drive

In [ ]:
import os, os.path as osp, tarfile, time

TGZ_PATH = '/content/drive/MyDrive/AI_Action/Data/v1.0-mini.tgz'  #@param {type:"string"}
DATAROOT = '/content/nuscenes'                              #@param {type:"string"}

if not osp.exists('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')

if osp.exists(osp.join(DATAROOT, 'v1.0-mini')):
    print('Da co du lieu, bo qua giai nen.')
else:
    assert osp.exists(TGZ_PATH), f'Khong thay {TGZ_PATH}'
    os.makedirs(DATAROOT, exist_ok=True)
    t0 = time.time()
    print('Dang giai nen...')
    with tarfile.open(TGZ_PATH, 'r:gz') as tar:
        tar.extractall(DATAROOT)
    print(f'Xong sau {time.time() - t0:.0f}s')

from nuscenes.nuscenes import NuScenes
nusc = NuScenes(version='v1.0-mini', dataroot=DATAROOT, verbose=False)
print('Nap xong:', len(nusc.scene), 'scene,', len(nusc.sample), 'keyframe')

## Cell 5 — Tải trọng số PointPillars đã huấn luyện

Dùng `mim download` để lấy đúng file cấu hình và trọng số khớp với phiên bản mmdet3d đang cài. Mô hình: **PointPillars + FPN**, huấn luyện trên tập train đầy đủ của nuScenes (700 scene), 10 lớp.

In [ ]:
import glob

CKPT_DIR = '/content/checkpoints'
MODEL_NAME = 'pointpillars_hv_fpn_sbn-all_8xb4-2x_nus-3d'

if not glob.glob(f'{CKPT_DIR}/*.pth'):
    !mim download mmdet3d --config {MODEL_NAME} --dest {CKPT_DIR}

cfg_file  = glob.glob(f'{CKPT_DIR}/*.py')[0]
ckpt_file = glob.glob(f'{CKPT_DIR}/*.pth')[0]
print('config    :', cfg_file)
print('checkpoint:', ckpt_file)

## Cell 6 — Khởi tạo model

Cấu hình gốc của nuScenes gộp 10 sweep (`LoadPointsFromMultiSweeps`), việc đó cần thông tin sweep từ file annotation của mmdet3d. Ở đây ta suy luận trực tiếp trên **một lần quét lidar**, nên phải bỏ bước đó khỏi pipeline và tự tạo point cloud 5 chiều `(x, y, z, intensity, Δt)` với `Δt = 0`.

In [ ]:
from mmdet3d.apis import init_model, inference_detector

model = init_model(cfg_file, ckpt_file, device='cuda:0')

# Bo buoc gop nhieu sweep khoi pipeline suy luan.
pipeline = model.cfg.test_dataloader.dataset.pipeline
pipeline = [t for t in pipeline if t['type'] != 'LoadPointsFromMultiSweeps']
for t in pipeline:
    if t['type'] == 'LoadPointsFromFile':
        t['load_dim'], t['use_dim'] = 5, 5
model.cfg.test_dataloader.dataset.pipeline = pipeline

CLASSES = list(model.dataset_meta['classes'])
print('10 lop model du doan:', CLASSES)
print('\nPipeline suy luan:', [t['type'] for t in pipeline])

## Cell 7 — Các hàm xử lý

Bốn hàm:
- `prepare_points()` — đọc file lidar của devkit, ghi lại thành file 5 chiều mà model cần
- `run_pointpillars()` — chạy suy luận, lọc theo ngưỡng điểm tin cậy
- `load_ground_truth()` — lấy nhãn gốc, **chỉ giữ 10 lớp được chấm điểm**
- `match_boxes()` — so khớp theo chuẩn nuScenes: cùng lớp, tâm cách nhau ≤ 2 m

Cả dự đoán lẫn nhãn gốc đều nằm trong **hệ toạ độ lidar**, nên so sánh được trực tiếp. Để tránh sai lệch quy ước góc xoay giữa hai thư viện, phần vẽ dùng thẳng **8 góc hộp** do mỗi bên tự tính, còn phần so khớp chỉ dùng **toạ độ tâm**.

In [ ]:
import numpy as np
from nuscenes.eval.detection.utils import category_to_detection_name

TMP_BIN = '/content/_frame.bin'


def get_sample(scene_name, index):
    scene = next(s for s in nusc.scene if s['name'] == scene_name)
    assert index < scene['nbr_samples'], \
        f"{scene_name} chi co {scene['nbr_samples']} keyframe"
    token = scene['first_sample_token']
    for _ in range(index):
        token = nusc.get('sample', token)['next']
    return nusc.get('sample', token), scene


def prepare_points(sample):
    """File lidar cua devkit: (x, y, z, intensity, ring). Model can (x, y, z, intensity, dt)."""
    path = nusc.get_sample_data_path(sample['data']['LIDAR_TOP'])
    pts = np.fromfile(path, dtype=np.float32).reshape(-1, 5).copy()
    pts[:, 4] = 0.0                      # dt = 0 vi chi dung 1 sweep
    pts.tofile(TMP_BIN)
    return pts, TMP_BIN


def run_pointpillars(sample, score_thr=0.3):
    """Tra ve dict: corners (N,8,3), centers (N,3), names, scores."""
    _, bin_path = prepare_points(sample)
    result, _ = inference_detector(model, bin_path)
    pred = result.pred_instances_3d

    scores = pred.scores_3d.cpu().numpy()
    keep = scores >= score_thr
    boxes = pred.bboxes_3d[keep]
    labels = pred.labels_3d.cpu().numpy()[keep]
    return {
        'corners': boxes.corners.cpu().numpy(),          # (N, 8, 3) he toa do lidar
        'centers': boxes.gravity_center.cpu().numpy(),   # (N, 3) tam hinh hoc
        'names': [CLASSES[i] for i in labels],
        'scores': scores[keep],
    }


def load_ground_truth(sample, min_pts=1):
    """Nhan goc, quy ve 10 lop duoc cham diem. min_pts=1 loai box khong co diem lidar."""
    _, boxes, _ = nusc.get_sample_data(sample['data']['LIDAR_TOP'])
    corners, centers, names = [], [], []
    for b in boxes:
        name = category_to_detection_name(b.name)
        if name is None:                 # vd bicycle_rack, animal: khong cham diem
            continue
        ann = nusc.get('sample_annotation', b.token)
        if ann['num_lidar_pts'] < min_pts:
            continue                     # chuan nuScenes: bo vat khong co diem lidar
        corners.append(b.corners().T)
        centers.append(b.center)
        names.append(name)
    return {
        'corners': np.array(corners).reshape(-1, 8, 3),
        'centers': np.array(centers).reshape(-1, 3),
        'names': names,
    }


def match_boxes(pred, gt, dist_th=2.0):
    """Khop tham lam theo diem tin cay giam dan; dung khoang cach tam tren mat phang ngang."""
    n_p, n_g = len(pred['names']), len(gt['names'])
    pred_tp = np.zeros(n_p, dtype=bool)
    gt_hit = np.zeros(n_g, dtype=bool)
    pairs = []

    for pi in np.argsort(-pred['scores']):
        best, best_d = -1, dist_th
        for gi in range(n_g):
            if gt_hit[gi] or gt['names'][gi] != pred['names'][pi]:
                continue
            d = np.linalg.norm(pred['centers'][pi][:2] - gt['centers'][gi][:2])
            if d < best_d:
                best, best_d = gi, d
        if best >= 0:
            pred_tp[pi], gt_hit[best] = True, True
            pairs.append((pi, best, best_d))
    return pred_tp, gt_hit, pairs


print('Da dinh nghia xong cac ham.')

## Cell 8 — Hàm vẽ hai hình cạnh nhau

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

EDGES = [(0, 1), (1, 2), (2, 3), (3, 0), (4, 5), (5, 6), (6, 7), (7, 4),
         (0, 4), (1, 5), (2, 6), (3, 7)]


def _box_lines(corners_list, color, label, texts=None):
    """Gom nhieu hop vao mot trace duy nhat cho nhe."""
    xs, ys, zs = [], [], []
    for k in corners_list:
        for i, j in EDGES:
            xs += [k[i, 0], k[j, 0], None]
            ys += [k[i, 1], k[j, 1], None]
            zs += [k[i, 2], k[j, 2], None]
    return go.Scatter3d(x=xs, y=ys, z=zs, mode='lines',
                        line=dict(color=color, width=3),
                        name=f'{label} ({len(corners_list)})',
                        hoverinfo='name' if texts is None else 'text',
                        text=texts)


def _points_trace(pts, max_points=40000):
    if len(pts) > max_points:
        pts = pts[np.random.default_rng(0).choice(len(pts), max_points, replace=False)]
    return go.Scatter3d(x=pts[:, 0], y=pts[:, 1], z=pts[:, 2], mode='markers',
                        marker=dict(size=1, color=pts[:, 2], colorscale='Greys',
                                    opacity=0.55, showscale=False),
                        name='lidar', hoverinfo='skip', showlegend=False)


def show_pred_vs_gt(scene_name='scene-0061', index=20, score_thr=0.3, dist_th=2.0,
                    height=750, out_html=None):
    sample, scene = get_sample(scene_name, index)
    pts, _ = prepare_points(sample)
    gt = load_ground_truth(sample)
    pred = run_pointpillars(sample, score_thr)
    pred_tp, gt_hit, pairs = match_boxes(pred, gt, dist_th)

    n_tp, n_fp, n_fn = int(pred_tp.sum()), int((~pred_tp).sum()), int((~gt_hit).sum())
    precision = n_tp / max(len(pred['names']), 1)
    recall = n_tp / max(len(gt['names']), 1)

    fig = make_subplots(rows=1, cols=2, specs=[[{'type': 'scene'}, {'type': 'scene'}]],
                        subplot_titles=(f'NHAN GOC — {len(gt["names"])} box',
                                        f'DU DOAN PointPillars — {len(pred["names"])} box'))

    for col in (1, 2):
        fig.add_trace(_points_trace(pts), row=1, col=col)

    # Ben trai: nhan goc
    hit = [c for c, h in zip(gt['corners'], gt_hit) if h]
    miss = [c for c, h in zip(gt['corners'], gt_hit) if not h]
    if hit:
        fig.add_trace(_box_lines(hit, 'limegreen', 'GT duoc tim ra'), row=1, col=1)
    if miss:
        fig.add_trace(_box_lines(miss, 'orange', 'GT bi bo sot (FN)',
                                 texts=[n for n, h in zip(gt['names'], gt_hit) if not h]),
                      row=1, col=1)

    # Ben phai: du doan
    tp = [c for c, t in zip(pred['corners'], pred_tp) if t]
    fp = [c for c, t in zip(pred['corners'], pred_tp) if not t]
    if tp:
        fig.add_trace(_box_lines(tp, 'limegreen', 'Du doan dung (TP)'), row=1, col=2)
    if fp:
        fig.add_trace(_box_lines(fp, 'red', 'Bao nham (FP)',
                                 texts=[f'{n} {s:.2f}' for n, s, t in
                                        zip(pred['names'], pred['scores'], pred_tp) if not t]),
                      row=1, col=2)

    cam = dict(eye=dict(x=-1.3, y=-1.3, z=0.9))
    fig.update_layout(
        template='plotly_dark', height=height,
        title=(f"{scene['name']} keyframe {index} · nguong diem {score_thr} · "
               f"TP {n_tp} · FP {n_fp} · FN {n_fn} · "
               f"precision {precision:.2f} · recall {recall:.2f}"),
        scene=dict(aspectmode='data', camera=cam),
        scene2=dict(aspectmode='data', camera=cam),
        margin=dict(l=0, r=0, t=70, b=0))

    if out_html:
        fig.write_html(out_html)
        print('Da luu', out_html)
    fig.show()
    return dict(gt=gt, pred=pred, pred_tp=pred_tp, gt_hit=gt_hit, pairs=pairs,
                tp=n_tp, fp=n_fp, fn=n_fn, precision=precision, recall=recall)


print('Da dinh nghia xong ham ve.')

## Cell 9 — Chạy và xem kết quả

Hai khung 3D **xoay độc lập** với nhau. Bấm vào chú thích để ẩn/hiện từng nhóm box.

In [ ]:
SCENE     = 'scene-0061'  #@param ['scene-0061','scene-0103','scene-0553','scene-0655','scene-0757','scene-0796','scene-0916','scene-1077','scene-1094','scene-1100']
KEYFRAME  = 20            #@param {type:"slider", min:0, max:38, step:1}
SCORE_THR = 0.3           #@param {type:"slider", min:0.05, max:0.9, step:0.05}

res = show_pred_vs_gt(SCENE, KEYFRAME, score_thr=SCORE_THR)

## Cell 10 — Bảng chi tiết theo từng lớp

In [ ]:
import collections

rows = collections.defaultdict(lambda: [0, 0, 0])   # [TP, FP, FN]
for n, t in zip(res['pred']['names'], res['pred_tp']):
    rows[n][0 if t else 1] += 1
for n, h in zip(res['gt']['names'], res['gt_hit']):
    if not h:
        rows[n][2] += 1

print(f"{'lop':22s} {'TP':>4s} {'FP':>4s} {'FN':>4s}  {'precision':>9s} {'recall':>7s}")
print('-' * 60)
for n in sorted(rows, key=lambda k: -sum(rows[k])):
    tp, fp, fn = rows[n]
    p = tp / (tp + fp) if tp + fp else float('nan')
    r = tp / (tp + fn) if tp + fn else float('nan')
    print(f'{n:22s} {tp:4d} {fp:4d} {fn:4d}  {p:9.2f} {r:7.2f}')
print('-' * 60)
print(f"{'TONG':22s} {res['tp']:4d} {res['fp']:4d} {res['fn']:4d}  "
      f"{res['precision']:9.2f} {res['recall']:7.2f}")

print('\nCac cap khop duoc (chi so du doan, chi so nhan goc, khoang cach m):')
for pi, gi, d in sorted(res['pairs'], key=lambda x: x[2])[:10]:
    print(f"  {res['pred']['names'][pi]:20s} diem {res['pred']['scores'][pi]:.2f}  cach {d:.2f} m")

## Cell 11 — Ảnh hưởng của ngưỡng điểm tin cậy

Ngưỡng thấp: tìm được nhiều hơn nhưng báo nhầm nhiều. Ngưỡng cao: ít báo nhầm nhưng bỏ sót nhiều.

In [ ]:
sample, _ = get_sample(SCENE, KEYFRAME)
gt = load_ground_truth(sample)
pred_all = run_pointpillars(sample, score_thr=0.0)

print(f"{'nguong':>7s} {'so box':>7s} {'TP':>4s} {'FP':>4s} {'FN':>4s} {'precision':>10s} {'recall':>7s}")
print('-' * 55)
for thr in [0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]:
    keep = pred_all['scores'] >= thr
    sub = {'corners': pred_all['corners'][keep], 'centers': pred_all['centers'][keep],
           'names': [n for n, k in zip(pred_all['names'], keep) if k],
           'scores': pred_all['scores'][keep]}
    tp_mask, hit, _ = match_boxes(sub, gt)
    tp, fp, fn = int(tp_mask.sum()), int((~tp_mask).sum()), int((~hit).sum())
    p = tp / max(len(sub['names']), 1)
    r = tp / max(len(gt['names']), 1)
    print(f'{thr:7.2f} {len(sub["names"]):7d} {tp:4d} {fp:4d} {fn:4d} {p:10.2f} {r:7.2f}')

## Cell 12 (tuỳ chọn) — Lưu HTML và tải về

In [ ]:
show_pred_vs_gt(SCENE, KEYFRAME, score_thr=SCORE_THR, out_html='pred_vs_gt.html')

from google.colab import files
files.download('pred_vs_gt.html')

---
## Cách đọc kết quả

| Màu | Ở hình bên trái (nhãn gốc) | Ở hình bên phải (dự đoán) |
|---|---|---|
| 🟢 Xanh lá | Vật thể model đã tìm ra | Dự đoán đúng (TP) |
| 🟠 Cam | Vật thể model **bỏ sót** (FN) | — |
| 🔴 Đỏ | — | **Báo nhầm** (FP) |

**Cách so khớp** theo chuẩn nuScenes: một dự đoán được tính là đúng khi **cùng lớp** với nhãn gốc và **tâm cách nhau ≤ 2 m** trên mặt phẳng ngang. Khác với detection 2D, nuScenes không dùng IoU mà dùng khoảng cách tâm, vì vật nhỏ ở xa rất khó đạt IoU cao dù vị trí đã đúng.

## Vì sao kết quả sẽ không hoàn hảo

- **Chỉ dùng 1 sweep.** Model gốc được huấn luyện với point cloud gộp 10 sweep (dày gấp ~5 lần). Dùng 1 sweep khiến vật ở xa có quá ít điểm, nên recall sẽ thấp hơn con số công bố.
- **Không dự đoán được vận tốc** khi chỉ có một khung hình.
- **`barrier` và `traffic_cone` rất khó**: chúng nhỏ, và trong cảnh công trường như scene-0061 chúng đứng sát nhau thành hàng dài.
- Nhãn gốc ở đây đã **loại các box không có điểm lidar nào**, đúng như cách nuScenes chấm điểm chính thức.

## Xử lý sự cố

| Lỗi | Cách xử lý |
|---|---|
| `AssertionError: MMCV==2.2.0 is used but incompatible` | Chạy lại cell vá ở Cell 2, rồi restart |
| `No module named 'mmdet3d'` | Chưa restart sau khi cài. Runtime → Restart session |
| `CUDA out of memory` | Runtime → Restart, rồi chạy lại từ Cell 3 |
| `KeyError: 'sweeps'` | Cell 6 chưa chạy (chưa bỏ `LoadPointsFromMultiSweeps` khỏi pipeline) |
| `mim download` lỗi mạng | Chạy lại cell, hoặc tải thủ công từ model zoo của MMDetection3D |
| Cài đặt hỏng hoàn toàn | Runtime → Disconnect and delete runtime, rồi chạy lại từ Cell 1 |

## Ghi chú

Trọng số được huấn luyện trên **700 scene train** của nuScenes đầy đủ. 10 scene trong bản mini **nằm trong tập train đó**, nên kết quả ở đây **lạc quan hơn thực tế** — đừng coi đây là đánh giá khách quan, nó chỉ để quan sát model hoạt động thế nào.